# 01 Random Walk and GBM

このNotebookでは、FinSimLabの最初のシミュレーション機能を使って、ランダムウォーク、幾何ブラウン運動、平均回帰過程を観察します。

目的は投資判断ではなく、パラメータを変えたときに確率過程の形がどう変わるかを直感的に見ることです。

## セットアップ

ローカル開発中は、リポジトリのルートで次を実行してからNotebookを開きます。

```bash
pip install -e ".[notebook]"
```

In [ ]:
import matplotlib.pyplot as plt
import finsimlab as fsl

plt.style.use("seaborn-v0_8-whitegrid")

## ランダムウォーク

ランダムウォークは、毎ステップの小さな変化を積み上げるモデルです。価格過程そのものとしては単純ですが、確率的な変動を学ぶ入り口として便利です。

In [ ]:
random_walk = fsl.simulate_random_walk(
    initial_value=0,
    drift=0.02,
    volatility=1.0,
    steps=100,
    n_paths=30,
    seed=42,
)

ax = fsl.plot_paths(random_walk, title="Random walk")
plt.show()

## 幾何ブラウン運動

幾何ブラウン運動は、入門的な株価モデルとしてよく使われます。価格が負にならない点がランダムウォークとの大きな違いです。

In [ ]:
gbm = fsl.simulate_gbm(
    s0=100,
    mu=0.05,
    sigma=0.2,
    years=1,
    steps=252,
    n_paths=50,
    seed=42,
)

ax = fsl.plot_paths(gbm, title="Geometric Brownian motion", ylabel="Price")
plt.show()

## 平均回帰過程

平均回帰過程は、値が長期平均へ戻ろうとする動きを表します。金利やスプレッドなど、平均へ戻る性質を仮定したい対象の入門モデルとして使えます。

In [ ]:
mean_reversion = fsl.simulate_mean_reversion(
    initial_value=3.0,
    long_term_mean=1.0,
    speed=2.0,
    volatility=0.4,
    years=2,
    steps=252,
    n_paths=30,
    seed=42,
)

ax = fsl.plot_paths(mean_reversion, title="Mean-reverting process")
ax.axhline(1.0, color="tab:red", linestyle="--", linewidth=1.5, label="Long-term mean")
ax.legend()
plt.show()

## スライダーで動かす

`ipywidgets` が入っている環境では、次のセルでGBMのパラメータを動かせます。Notebookでも簡単なUIを作れるため、Pythonライブラリとグラフィカルな学習体験は両立できます。

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def interactive_gbm(mu=0.05, sigma=0.2, n_paths=30):
        paths = fsl.simulate_gbm(
            s0=100,
            mu=mu,
            sigma=sigma,
            years=1,
            steps=252,
            n_paths=n_paths,
            seed=7,
        )
        ax = fsl.plot_paths(paths, title="Interactive GBM", ylabel="Price")
        plt.show()

    display(
        widgets.interactive(
            interactive_gbm,
            mu=widgets.FloatSlider(value=0.05, min=-0.2, max=0.2, step=0.01),
            sigma=widgets.FloatSlider(value=0.2, min=0.0, max=0.6, step=0.01),
            n_paths=widgets.IntSlider(value=30, min=1, max=100, step=1),
        )
    )
except ImportError:
    print("ipywidgets is not installed. Install with: pip install -e '.[notebook]'")

## 学習メモ: 直感・数式・演習

### 直感

ランダムウォークは「毎回少しずつ上下に動く」だけの単純なモデルです。1本の経路だけを見ると偶然に強く左右されますが、多数の経路を重ねると、ドリフトが平均的な方向、ボラティリティがばらつきの大きさを決めていることが見えてきます。

幾何ブラウン運動（GBM）は、価格がマイナスにならないように変化率を積み上げるモデルです。株価そのものではなく、リターンがランダムに動くと考えると理解しやすくなります。

平均回帰過程は、値が長期平均から離れるほど戻る力が強くなるモデルです。短期金利やスプレッドのように、一定の水準へ戻る性質を仮定したいときの入口になります。

### 数式の最小説明

単純なランダムウォークは、前の値に小さな変化を足していく形です。

$$X_{t+1} = X_t + \mu \Delta t + \sigma \epsilon_t$$

GBMでは、価格の変化率にランダム性を入れます。

$$dS_t = \mu S_t dt + \sigma S_t dW_t$$

ここで、$\mu$ は平均的な成長率、$\sigma$ は変動の大きさ、$dW_t$ はランダムな揺れを表します。FinSimLabでは教育目的の簡易シミュレーションとして使います。

### 演習問題

1. GBMの `sigma` を `0.1`, `0.2`, `0.4` に変えて、価格パスの広がり方を比較してください。
2. `mu` をマイナスにしたとき、1本の経路と多数の経路の平均がどう変わるか確認してください。
3. 平均回帰過程の `speed` を大きくすると、長期平均へ戻る速さがどう変わるか観察してください。
4. `n_paths` を増やすと、見た目のばらつきの理解がどう変わるか試してください。

> 注意: ここで扱うモデルは学習用の単純化です。実際の市場価格を予測するものではありません。
